# 10 — Agentic RAG: Let an Agent Choose Retrieval Tools

## Goal
Create a small agentic RAG workflow in which a tool-using agent decides when to search the CIS Windows 11 benchmark, can run more than one search, and synthesizes an answer from tool results.

This notebook uses LangChain's `create_agent` API. Tool calls are read-only: they search the local Chroma store and return source snippets. The agent does not change endpoint settings or execute commands on Windows.

**Prerequisite:** Run `00b_ingestion_vector_store.ipynb` first. Ensure your installed LangChain version includes `langchain.agents.create_agent`.

## Architecture

```mermaid
flowchart TD
    A["User question"] --> B["Agent / LLM"]
    B --> C{"Need evidence?"}
    C -->|"Yes"| D["Search benchmark tool"]
    D --> E["Return chunks + source metadata"]
    E --> B
    C -->|"Enough evidence"| F["Synthesize grounded answer"]
    F --> G["Answer with source references"]

    classDef input fill:#dbeafe,stroke:#2563eb,color:#172554,stroke-width:2px;
    classDef agent fill:#fef3c7,stroke:#d97706,color:#78350f,stroke-width:2px;
    classDef tool fill:#dcfce7,stroke:#16a34a,color:#14532d,stroke-width:2px;
    classDef decision fill:#fce7f3,stroke:#db2777,color:#831843,stroke-width:2px;
    classDef output fill:#ede9fe,stroke:#7c3aed,color:#3b0764,stroke-width:2px;
    class A,G input;
    class B,F agent;
    class D,E tool;
    class C decision;
```

## 1. Imports and paths

In [ ]:
from pathlib import Path
import re
from collections import OrderedDict

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "vector_store").exists() and (PROJECT_ROOT.parent / "vector_store").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

VECTOR_STORE_DIR = PROJECT_ROOT / "vector_store" / "chroma_cis_windows11"
COLLECTION_NAME = "cis_windows11_benchmark"

print("Project root:", PROJECT_ROOT)
print("Vector store exists:", VECTOR_STORE_DIR.exists())

## 2. Select a model provider

In [ ]:
PROVIDER = "ollama"  # "ollama" or "groq"
OLLAMA_MODEL = "gemma4:e2b"
GROQ_MODEL = "llama-3.3-70b-versatile"

if PROVIDER == "ollama":
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)
    print(f"Using Ollama model: {OLLAMA_MODEL}")
elif PROVIDER == "groq":
    from langchain_groq import ChatGroq
    key_path = PROJECT_ROOT / "secrets" / "groq_api_key.txt"
    if not key_path.exists():
        raise FileNotFoundError(
            f"Groq key file not found: {key_path}. Create it and put only your API key in it."
        )
    groq_api_key = key_path.read_text(encoding="utf-8").strip()
    if not groq_api_key:
        raise ValueError(f"Groq key file is empty: {key_path}")
    llm = ChatGroq(model=GROQ_MODEL, api_key=groq_api_key, temperature=0)
    print(f"Using Groq model: {GROQ_MODEL}")
else:
    raise ValueError("PROVIDER must be 'ollama' or 'groq'")

## 3. Open the persisted Chroma store

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
if not VECTOR_STORE_DIR.exists():
    raise FileNotFoundError(
        f"Vector store not found at {VECTOR_STORE_DIR}. "
        "Run notebooks/00b_ingestion_vector_store.ipynb first."
    )

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=str(VECTOR_STORE_DIR),
    embedding_function=embedding_model,
)
print("Connected to:", COLLECTION_NAME)
print("Stored record count:", vector_store._collection.count())

## 4. Define read-only search tools

In [ ]:
from langchain_core.tools import tool

@tool
def search_windows_benchmark(query: str) -> str:
    """Search the CIS Windows 11 Enterprise benchmark for relevant security guidance. Use for specific configuration recommendations."""
    docs = vector_store.similarity_search(query, k=5)
    if not docs:
        return "No matching benchmark chunks were retrieved."
    blocks = []
    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata or {}
        source = Path(str(meta.get("source", "unknown source"))).name
        page = meta.get("page", "unknown")
        start = meta.get("start_index")
        label = f"[S{i}] {source}, page {page}"
        if start is not None:
            label += f", chunk start {start}"
        blocks.append(f"{label}\\n{doc.page_content[:1600]}")
    return "\\n\\n---\\n\\n".join(blocks)

@tool
def search_benchmark_for_verification(query: str) -> str:
    """Run a second, broader benchmark search when the first search does not provide enough detail."""
    broad_query = (
        "CIS Microsoft Windows 11 Enterprise Benchmark recommendation, rationale, "
        "audit and remediation guidance for: " + query
    )
    docs = vector_store.similarity_search(broad_query, k=5)
    if not docs:
        return "No matching benchmark chunks were retrieved on the broader search."
    blocks = []
    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata or {}
        source = Path(str(meta.get("source", "unknown source"))).name
        page = meta.get("page", "unknown")
        blocks.append(f"[B{i}] {source}, page {page}\\n{doc.page_content[:1600]}")
    return "\\n\\n---\\n\\n".join(blocks)

print("Read-only tools ready:", search_windows_benchmark.name, search_benchmark_for_verification.name)

## 5. Create the agent

The agent can decide which search tool to call and whether another search is needed. The system instructions restrict answers to retrieved benchmark evidence and prohibit claims about the current compliance state of a machine.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[search_windows_benchmark, search_benchmark_for_verification],
    system_prompt=(
        "You are a careful assistant for the CIS Microsoft Windows 11 Enterprise Benchmark. "
        "Use the search tools to ground security recommendation answers. For broad or multi-part "
        "questions, search for each important aspect or use the broader verification tool when "
        "the first result is insufficient. Cite source labels returned by tools, such as [S1] or [B1]. "
        "If the benchmark evidence is insufficient, say so. The benchmark describes recommendations; "
        "it does not establish a real endpoint's current settings or compliance. Never claim that it does. "
        "Tools are read-only; do not claim to change system settings."
    ),
)
print("Agent created.")

## 6. Run the agent

The result structure can vary a little by LangChain version. This cell prints the final message and also leaves the full state available for inspection.

In [ ]:
question = "What does the benchmark recommend about screen locking, and what should be checked to verify a machine's setting?"
agent_result = agent.invoke({
    "messages": [{"role": "user", "content": question}]
})

messages = agent_result.get("messages", [])
print("Messages in agent state:", len(messages))
if messages:
    final_message = messages[-1]
    print("\\nFINAL MESSAGE\\n")
    print(getattr(final_message, "content", final_message))
else:
    print(agent_result)

## 7. Inspect the agent's tool activity

In [ ]:
for i, message in enumerate(agent_result.get("messages", [])):
    print("=" * 88)
    print("Message", i, "| type:", type(message).__name__)
    print("Content:", getattr(message, "content", "")[:1800])
    tool_calls = getattr(message, "tool_calls", None)
    if tool_calls:
        print("Tool calls:", tool_calls)

## 8. Try another question

Agentic behavior is most visible when the question has multiple aspects. Inspect the tool calls to see whether the agent performs one or more searches.

In [ ]:
question2 = "Summarize benchmark guidance related to password policy and account lockout."
agent_result2 = agent.invoke({
    "messages": [{"role": "user", "content": question2}]
})
messages2 = agent_result2.get("messages", [])
if messages2:
    print(getattr(messages2[-1], "content", messages2[-1]))
print("\\nTool activity:")
for message in messages2:
    tool_calls = getattr(message, "tool_calls", None)
    if tool_calls:
        for call in tool_calls:
            print(call.get("name"), call.get("args"))

## Summary and limitations

- The LLM chooses when and how to call read-only retrieval tools.
- Tool use can make the pipeline more flexible for multi-part questions.
- Agent behavior is nondeterministic and may stop early, repeat searches, or misinterpret evidence.
- Always inspect tool calls and cited chunks; citations generated by the model must correspond to returned source labels.
- Keep tool permissions narrow. This workshop agent searches documents only and cannot configure Windows.
- Benchmark recommendations do not establish current endpoint compliance.